# Regression Prediction: MeanRegressor on eval_users1000

Small summary notebook for the MovieLens-1M rating-regression baseline. It reads compact run provenance from `outputs/in_distribution/regression_prediction/`, keeps the latest complete artifact per `(task, method)`, and reports seed-level and seed-averaged metrics.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "beyond-click-sim":
            return candidate
    raise RuntimeError("Could not find beyond-click-sim repo root")

REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "outputs" / "in_distribution" / "regression_prediction"
RESULTS_ROOT

PosixPath('/Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim/outputs/in_distribution/regression_prediction')

In [2]:
TASK_RE = re.compile(
    r"^(?P<dataset>.+)_(?P<target>[^_]+)_eval_users(?P<eval_users>\d+)_seed(?P<seed>\d+)$"
)
RUN_TS_RE = re.compile(r"^(?P<timestamp>\d{8}T\d{6}Z)_")

def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

def timestamp_from_run_dir(run_dir: Path) -> str:
    match = RUN_TS_RE.match(run_dir.name)
    return "" if match is None else match.group("timestamp")

def parse_task_name(task: str) -> dict[str, str | int]:
    match = TASK_RE.match(task)
    if match is None:
        return {"dataset": "unknown", "target": "unknown", "eval_users": -1, "seed": -1}
    data = match.groupdict()
    return {
        "dataset": data["dataset"],
        "target": data["target"],
        "eval_users": int(data["eval_users"]),
        "seed": int(data["seed"]),
    }

def collect_regression_runs() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(RESULTS_ROOT.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        manifest_path = run_dir / "manifest.json"
        if not manifest_path.exists():
            continue
        metrics = read_json(metrics_path)
        manifest = read_json(manifest_path)
        if metrics.get("protocol") != "regression":
            continue
        task = metrics["task"]
        task_info = parse_task_name(task)
        task_manifest = manifest["task"]["manifest"]
        rows.append({
            "timestamp": timestamp_from_run_dir(run_dir),
            "run": run_dir.name,
            "task": task,
            "method": metrics["method"],
            "dataset": task_info["dataset"],
            "target": task_info["target"],
            "eval_users": task_info["eval_users"],
            "seed": task_info["seed"],
            "train_rows": task_manifest["rows"]["train"],
            "val_rows": task_manifest["rows"]["val"],
            "test_rows": task_manifest["rows"]["test"],
            "train_users": task_manifest["users"]["train"],
            "val_users": task_manifest["users"]["val"],
            "test_users": task_manifest["users"]["test"],
            "scorer_mean": manifest["scorer"]["mean"],
            "val_macro_mae": metrics["val"]["macro_by_user_mean"]["mae"],
            "val_macro_rmse": metrics["val"]["macro_by_user_mean"]["rmse"],
            "test_macro_mae": metrics["test"]["macro_by_user_mean"]["mae"],
            "test_macro_rmse": metrics["test"]["macro_by_user_mean"]["rmse"],
            "test_micro_mae": metrics["test"]["micro"]["mae"],
            "test_micro_rmse": metrics["test"]["micro"]["rmse"],
            "main_metric": metrics["main_metric"],
        })
    return pd.DataFrame(rows)

all_runs = collect_regression_runs()
latest_runs = (
    all_runs.sort_values("timestamp")
    .groupby(["task", "method"], as_index=False, dropna=False)
    .tail(1)
    .sort_values(["dataset", "target", "method", "seed"])
    .reset_index(drop=True)
)
latest_runs[["dataset", "target", "method", "seed", "timestamp", "run"]]

,dataset,target,method,seed,timestamp,run
0,ml-1m,rating,mean_regressor,0,20260612T121634Z,20260612T121634Z_ml-1m_rating_eval_users1000_s...
1,ml-1m,rating,mean_regressor,1,20260612T121644Z,20260612T121644Z_ml-1m_rating_eval_users1000_s...
2,ml-1m,rating,mean_regressor,2,20260612T121654Z,20260612T121654Z_ml-1m_rating_eval_users1000_s...
3,ml-1m,rating,mean_regressor,3,20260612T121705Z,20260612T121705Z_ml-1m_rating_eval_users1000_s...
4,ml-1m,rating,mean_regressor,4,20260612T121716Z,20260612T121716Z_ml-1m_rating_eval_users1000_s...


## Seed Runs

In [3]:
seed_table = latest_runs[[
    "dataset", "target", "method", "seed",
    "train_rows", "val_rows", "test_rows", "test_users",
    "scorer_mean", "test_macro_mae", "test_macro_rmse",
    "test_micro_mae", "test_micro_rmse", "run",
]]

display(
    seed_table.style
    .format({
        "scorer_mean": "{:.6f}",
        "test_macro_mae": "{:.4f}",
        "test_macro_rmse": "{:.4f}",
        "test_micro_mae": "{:.4f}",
        "test_micro_rmse": "{:.4f}",
    })
    .hide(axis="index")
)

dataset,target,method,seed,train_rows,val_rows,test_rows,test_users,scorer_mean,test_macro_mae,test_macro_rmse,test_micro_mae,test_micro_rmse,run
ml-1m,rating,mean_regressor,0,694335,17629,34520,1000,3.581205,0.9352,1.0809,0.9214,1.1009,20260612T121634Z_ml-1m_rating_eval_users1000_seed0_mean_regressor
ml-1m,rating,mean_regressor,1,694335,16574,32425,1000,3.581980,0.9330,1.0769,0.9352,1.1194,20260612T121644Z_ml-1m_rating_eval_users1000_seed1_mean_regressor
ml-1m,rating,mean_regressor,2,694335,16731,32738,1000,3.579498,0.9387,1.0832,0.9277,1.1107,20260612T121654Z_ml-1m_rating_eval_users1000_seed2_mean_regressor
ml-1m,rating,mean_regressor,3,694335,17388,34008,1000,3.580783,0.9394,1.0849,0.9246,1.1069,20260612T121705Z_ml-1m_rating_eval_users1000_seed3_mean_regressor
ml-1m,rating,mean_regressor,4,694335,17564,34407,1000,3.580365,0.9411,1.0860,0.9383,1.1219,20260612T121716Z_ml-1m_rating_eval_users1000_seed4_mean_regressor


## Seed-Averaged Summary

In [4]:
summary = (
    latest_runs.groupby(["dataset", "target", "method"], dropna=False)
    .agg(
        seeds=("seed", "nunique"),
        train_rows=("train_rows", "first"),
        train_users=("train_users", "first"),
        test_users=("test_users", "first"),
        test_macro_mae_mean=("test_macro_mae", "mean"),
        test_macro_mae_std=("test_macro_mae", "std"),
        test_macro_rmse_mean=("test_macro_rmse", "mean"),
        test_macro_rmse_std=("test_macro_rmse", "std"),
        test_micro_mae_mean=("test_micro_mae", "mean"),
        test_micro_mae_std=("test_micro_mae", "std"),
    )
    .reset_index()
)

display(
    summary.style
    .format({
        "test_macro_mae_mean": "{:.4f}",
        "test_macro_mae_std": "{:.4f}",
        "test_macro_rmse_mean": "{:.4f}",
        "test_macro_rmse_std": "{:.4f}",
        "test_micro_mae_mean": "{:.4f}",
        "test_micro_mae_std": "{:.4f}",
    })
    .hide(axis="index")
)

dataset,target,method,seeds,train_rows,train_users,test_users,test_macro_mae_mean,test_macro_mae_std,test_macro_rmse_mean,test_macro_rmse_std,test_micro_mae_mean,test_micro_mae_std
ml-1m,rating,mean_regressor,5,694335,6040,1000,0.9375,0.0033,1.0824,0.0036,0.9294,0.0071
